# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vincentoei/flyrank-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

Lane: Growth Prediction (freestyle)

I chose this lane because identifying pages likely to grow helps a team decide where to invest effort. Instead of only fixing problems, the model highlights opportunities (pages that already show positive momentum and may benefit from expansion, promotion, or protection). Growth prediction is harder than scoring current state because it requires a true future-window label, but it is more actionable for resource allocation.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Confirm lane choice and check that the starter data has the signals needed for Growth Prediction
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parents[1]
df = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")

print("Lane: Growth Prediction (freestyle)")
print(f"Starter dataset rows: {len(df):,}")
print(f"Clients: {df['client_id'].nunique()}")
print("Key signals available for this lane:")
for col in ["impressions_90d", "impressions_last_30d", "impressions_prev_30d",
            "avg_position", "content_age_days", "days_since_last_update", "ctr"]:
    print(f"  - {col}: present")
print(f"\nTrend directions (current snapshot):")
print(df["trend_direction"].value_counts(normalize=True).round(3))

# This prints the lane, the dataset size, and the available signals. It shows the lane is grounded in real data.


Lane: Growth Prediction (freestyle)
Starter dataset rows: 30,000
Clients: 32
Key signals available for this lane:
  - impressions_90d: present
  - impressions_last_30d: present
  - impressions_prev_30d: present
  - avg_position: present
  - content_age_days: present
  - days_since_last_update: present
  - ctr: present

Trend directions (current snapshot):
trend_direction
down      0.542
stable    0.199
up        0.146
new       0.075
flat      0.038
Name: proportion, dtype: float64


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Decision: A content team has limited time. They must decide which pages to expand, promote, protect, or monitor.

Action: The reviewer opens a ranked list of pages predicted to grow in the next 30 days, inspects the top candidates, and chooses between "expand content," "promote internally," "protect current ranking," or "monitor only."

Cost of a wrong call:
- False positive: effort is spent expanding a page that does not actually grow. The cost is wasted time and opportunity cost.
- False negative: a page with real growth potential is missed. The cost is unrealized traffic.

Why data/ML helps: Growth depends on multiple signals, which are recent acceleration, position trajectory, content age, freshness, engagement, and demand. A plain rule captures only one or two of these. A model can combine them and rank pages by growth likelihood.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Estimate the scale of the decision: how many pages currently look stable vs not,
# to show why prioritization matters.

import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parents[1]
df = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")

# Pages with enough volume to be worth reviewing
reviewable = df[df["impressions_90d"] >= 100]

print(f"Total pages: {len(df):,}")
print(f"Pages with >= 100 impressions in 90d: {len(reviewable):,}")
print(f"Share of inventory with reviewable volume: {len(reviewable)/len(df):.2%}")

print("\nDecision: which of these pages should the team prioritize?")
print("Action: inspect top-ranked pages and choose expand / promote / protect / monitor.")
print("Cost of wrong call: wasted reviewer time (false positive) or missed opportunity (false negative).")

# This shows the decision is real because there is a large pool of pages and limited reviewer capacity

Total pages: 30,000
Pages with >= 100 impressions in 90d: 22,006
Share of inventory with reviewable volume: 73.35%

Decision: which of these pages should the team prioritize?
Action: inspect top-ranked pages and choose expand / promote / protect / monitor.
Cost of wrong call: wasted reviewer time (false positive) or missed opportunity (false negative).


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

I loaded the starter CSV and checked the size, trend distribution, and a strict relative-growth proxy. The dataset has 30,000 pages from 32 clients. Only 14.6% are currently trending up, and only 7.8% meet a minimum-volume relative-growth proxy (prior 30 days ≥ 100 impressions and last 30 days at least 20% higher). This confirms that growth is a rare, valuable signal worth predicting.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parents[1]
df = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Clients: {df['client_id'].nunique()}")

print("\nTrend direction distribution:")
print(df["trend_direction"].value_counts(normalize=True).round(3))

print("\n90-day impressions (median):", df["impressions_90d"].median())
print("Content age in days (median):", df["content_age_days"].median())

# Strict relative-growth proxy on the starter snapshot
mask = (
    (df["impressions_prev_30d"] >= 100)
    & (df["impressions_last_30d"] >= 1.20 * df["impressions_prev_30d"])
)
print(f"\nPages meeting growth proxy: {mask.sum():,} ({mask.mean():.3%})")

# The median page has 731 impressions over 90 days and is 236 days old. Growth is rare: only 7.8% of pages meet the proxy. 
# This makes the lane worth pursuing, because a model that can rank pages by growth risk has clear decision value. 
# The warehouse release will let me build a true future-window label without leaking the current trend_direction




Rows: 30,000
Columns: 44
Clients: 32

Trend direction distribution:
trend_direction
down      0.542
stable    0.199
up        0.146
new       0.075
flat      0.038
Name: proportion, dtype: float64

90-day impressions (median): 731.0
Content age in days (median): 236.0

Pages meeting growth proxy: 2,354 (7.847%)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

What I can claim:
- "Pages with this score were more likely to show relative growth in impressions over the next 30 days."
- "This ranking helps a content team prioritize expansion, promotion, or protection."
- "The result is observed, directional, and decision-support."

What I cannot claim:
- "Promoting these pages will cause growth."
- "I predicted Google's algorithm."
- "The model proves causation."

Why: The data is observational. I can associate past signals with future movement, but I cannot prove that an action caused the movement. All claims will use careful language: "observed," "measured," "directional," "decision-support."

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Define safe and unsafe claim language for this project.
# This acts as a checklist you can scan your future write-up against.

safe_claims = [
    "Pages with this score were more likely to show relative growth in the next 30 days.",
    "This ranking helps a content team prioritize expansion or protection.",
    "The result is observed, directional, and decision-support.",
]

unsafe_claims = [
    "Promoting these pages will cause growth.",
    "I predicted Google's algorithm.",
    "The model proves causation.",
]

print("SAFE claims:")
for claim in safe_claims:
    print(f"  - {claim}")

print("\nUNSAFE claims:")
for claim in unsafe_claims:
    print(f"  - {claim}")

print("\nRule: every public claim must use 'observed', 'measured', 'directional', or 'decision-support'.")
print("No causal claims, no algorithm claims, no client-identifying details.")

# This backs up the careful-words section with a simple filter that can be reused later.

SAFE claims:
  - Pages with this score were more likely to show relative growth in the next 30 days.
  - This ranking helps a content team prioritize expansion or protection.
  - The result is observed, directional, and decision-support.

UNSAFE claims:
  - Promoting these pages will cause growth.
  - I predicted Google's algorithm.
  - The model proves causation.

Rule: every public claim must use 'observed', 'measured', 'directional', or 'decision-support'.
No causal claims, no algorithm claims, no client-identifying details.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.